# WiFall baseline on the shared split

Runs the WiFall adaptation (`analysis/baselines/wifall/`, see [the guide](../../docs/baselines/wifall.md)) with both of its classifiers, a balanced random forest and a fall-inlier one-class SVM, so that `06_comparison` can set it beside our sequence models.

Every approach is scored on the attempts that one `03_training` run held out
(`window_split.parquet`). The baselines train on that run's training and
validation attempts, because they choose their settings on training data only,
and are tested on its test attempts. Attempts that the baseline loads but the
split does not contain are left out of both sides and counted. Fall and
non-fall labels are read from the split, so the label mapping is the sequence
export's.

**Scope.** With `SAMPLE = True` in `02_preprocess` and `03_training`, the
split covers one session (Akmal, 9 November 13:15) and 2 s windows. Results on
it are a smoke check of the pipeline, not evidence of detection performance:
the session is small, informed the baseline adaptations, and comes from one
participant.

## Setup

Set `SPLIT_RUN` to a `03_training` run directory to pin the split; `None` uses the newest default-representation run that has one.

In [ ]:
import json
import sys
from dataclasses import replace
from datetime import UTC, datetime
from pathlib import Path

import polars as pl
from IPython.display import display

PROJECT_ROOT = next(
    p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src/fall_detection").is_dir()
)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fall_detection.analysis._cache_keys import file_hash
from fall_detection.analysis.baselines.wifall.dataset import load_windows
from fall_detection.analysis.baselines.wifall.experiment import (
    run_experiment,
    save_experiment,
)
from fall_detection.analysis.baselines.wifall.model import WiFallConfig
from fall_detection.analysis.comparison import (
    latest_training_run,
    split_assignment,
    split_labels,
    split_sessions,
)

# The held-out attempts come from a 03_training run; None selects the newest one.
SPLIT_RUN = None
TRAINING_RUNS = PROJECT_ROOT / "artifacts/notebooks/03_training"
split_run = SPLIT_RUN or latest_training_run(TRAINING_RUNS, "window_split.parquet")
split_path = split_run / "window_split.parquet"
split = pl.read_parquet(split_path)
training_manifest = json.loads((split_run / "manifest.json").read_text())
ASSIGNMENT = split_assignment(split)
FALL_LABELS, NON_FALL_LABELS = split_labels(split)
SOURCES = [
    PROJECT_ROOT / "data/raw/sessions" / f"{session}.zip"
    for session in split_sessions(split)
]
CLASSIFIERS = ("random_forest", "one_class_svm")
CONFIG = WiFallConfig(random_seed=42)
RUN = (
    PROJECT_ROOT
    / "artifacts/notebooks/04_wifall"
    / datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")
)
print(
    "Split:",
    split_run.relative_to(PROJECT_ROOT),
    "sample:",
    training_manifest.get("sample"),
)
print("Sessions:", [path.name for path in SOURCES])
print("Fall labels:", FALL_LABELS)
print("Non-fall labels:", NON_FALL_LABELS)

## Windows

Two-second windows sliding by 1 s inside each recording epoch, with receivers judged per window. The table counts windows and attempts per split and target; `unassigned` attempts are left out.

In [ ]:
windows = load_windows(
    SOURCES,
    root=PROJECT_ROOT,
    fall_labels=FALL_LABELS,
    non_fall_labels=NON_FALL_LABELS,
)
assigned = pl.DataFrame(
    {
        "group": windows.groups.tolist(),
        "target": windows.targets.tolist(),
    }
).with_columns(
    pl.col("group").replace_strict(ASSIGNMENT, default="unassigned").alias("split")
)
display(
    assigned.group_by("split", "target")
    .agg(pl.len().alias("windows"), pl.col("group").n_unique().alias("attempts"))
    .sort("split", "target")
)

## Train and test

Each classifier is fitted on the training windows (training and validation attempts) and scored on the test attempts. Outputs go to `artifacts/notebooks/04_wifall/<run>/<classifier>/`.

In [ ]:
RUN.mkdir(parents=True, exist_ok=False)
reports = {}
for classifier in CLASSIFIERS:
    result = run_experiment(
        windows, replace(CONFIG, classifier=classifier), assignment=ASSIGNMENT
    )
    save_experiment(result, RUN / classifier)
    reports[classifier] = result.report
(RUN / "manifest.json").write_text(
    json.dumps(
        {
            "notebook": "04_wifall",
            "split_run": str(split_run.relative_to(PROJECT_ROOT)),
            "split_sha256": file_hash(split_path),
            "training_manifest": training_manifest,
            "sources": [str(path.relative_to(PROJECT_ROOT)) for path in SOURCES],
            "fall_labels": list(FALL_LABELS),
            "non_fall_labels": list(NON_FALL_LABELS),
            "classifiers": list(CLASSIFIERS),
        },
        indent=2,
    )
    + "\n"
)
print("Results:", RUN.relative_to(PROJECT_ROOT))

## Held-out results

`window` is the full detector (LOF activity gate and classifier), `window, classifier only` skips the gate, and `attempt` counts an attempt as a predicted fall when any of its windows is.

In [ ]:
rows = []
for classifier, report in reports.items():
    for level, key in (
        ("window", "held_out"),
        ("window, classifier only", "held_out_classifier_only"),
        ("attempt", "held_out_attempts"),
    ):
        metrics = report[key]
        rows.append(
            {
                "classifier": classifier,
                "level": level,
                **{
                    name: metrics[name]
                    for name in (
                        "balanced_accuracy",
                        "precision",
                        "recall",
                        "f1",
                        "false_positive_rate",
                    )
                },
                "examples": sum(map(sum, metrics["confusion_matrix"])),
            }
        )
display(pl.DataFrame(rows))
print(
    "Test attempts:",
    len(reports[CLASSIFIERS[0]]["split"]["test_attempts"]),
    "unassigned attempts left out:",
    len(reports[CLASSIFIERS[0]]["split"]["unassigned_attempts"]),
)